# 精确f696原账户现金转SOXL合法动作空间

只读冻结0bp Tiered原路径、f696提案和自身风险投影，按30% SOXL、15%现金及0.10 beta诊断包络计算目标头寸容量。无费用重算、模型或候选资金账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_f696_cash_to_soxl_legal_headroom_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());s=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
a=root/s['A0_card'];b=root/s['account_card']
assert sha(a/'summary.json')==s['A0_summary_sha256'] and sha(b/'summary.json')==s['account_summary_sha256']
assert sha(root/s['provider']/'qlib_store_manifest.json')==s['provider_manifest_sha256']
sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text());ra=json.loads((root/s['A0_readback']).read_text())
assert sa['nas_evidence']['sha256']==s['A0_NAS_sha256'] and ra['status']=='passed' and ra['archive_sha256']==s['A0_NAS_sha256'] and ra['all_artifact_hashes_match']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==s['account_NAS_sha256']
for source,exp,rid,art,key in [(a,'A0_experiment_id','A0_recorder_id','native/A_active_research_replay_0','A0_native_sha256'),(a,'A0_experiment_id','A0_recorder_id','inputs/frame','A0_frame_sha256'),(a,'A0_experiment_id','A0_recorder_id','inputs/recovery_spec','A0_recovery_spec_sha256'),(b,'account_experiment_id','account_recorder_id','analysis/rolling126','account_rolling_sha256')]:
 assert sha(source/'qlib_recorder'/s[exp]/s[rid]/'artifacts'/art)==s[key],key
qlib.init(provider_uri=str((root/s['provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['A0_experiment_id'],recorder_id=s['A0_recorder_id'])
 native=rec.load_object('native/A_active_research_replay_0');frame=rec.load_object('inputs/frame');recovery=rec.load_object('inputs/recovery_spec')
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=s['account_experiment_id'],recorder_id=s['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
auth=native['authority'].reset_index(drop=True);targets=native['targets'].reset_index(drop=True);frame=frame.reset_index(drop=True)
for x in (auth,targets,frame):x['trade_date']=pd.to_datetime(x.trade_date).dt.normalize()
assert len(auth)==len(targets)==len(frame)==944 and len(rolling)==819
calendar=pd.DatetimeIndex(D.calendar(start_time='2023-01-04',end_time='2026-10-08',freq='day'))
assert len(calendar)==944 and pd.DatetimeIndex(auth.trade_date).equals(calendar)
assert auth.trade_date.equals(frame.trade_date) and auth.trade_date.equals(targets.trade_date)
assert int(auth.execute_open_target.sum())==797 and int((~auth.execute_open_target).sum())==147
mincash=float(recovery['minimum_cash_weight']);maxsoxl=float(recovery['maximum_soxl_weight'])
assert (mincash,maxsoxl)==(.15,.30)
raw_soxl=frame.soxl_weight.to_numpy(float);raw_soxx=frame.soxx_weight.to_numpy(float)
p_soxl=targets.soxl_weight.to_numpy(float);p_soxx=targets.soxx_weight.to_numpy(float);p_cash=targets.cash_weight.to_numpy(float)
cap=auth.current_beta_cap.to_numpy(float);execute=auth.execute_open_target.to_numpy(bool)
raw_beta=3*raw_soxl+raw_soxx;base_beta=3*p_soxl+p_soxx;binding=raw_beta>cap+1e-9
assert np.isfinite(np.column_stack([raw_soxl,raw_soxx,p_soxl,p_soxx,p_cash,cap])).all()
assert np.max(np.abs(p_soxl+p_soxx+p_cash-1))<1e-8 and (p_soxl<=maxsoxl+1e-9).all() and (p_cash>=mincash-1e-9).all()
assert (base_beta<=cap+1e-8).all()
ceiling=cap+np.where(binding,.10,0.)
beta_slack=np.maximum(0.,ceiling-base_beta)
cash_slack=3*np.maximum(0.,p_cash-mincash)
soxl_slack=3*np.maximum(0.,maxsoxl-p_soxl)
delta=np.where(execute,np.maximum(0.,np.minimum.reduce([np.full(944,.10),beta_slack,cash_slack,soxl_slack])),0.)
candidate_soxl=p_soxl+delta/3;candidate_cash=p_cash-delta/3;candidate_beta=base_beta+delta
assert (candidate_soxl<=maxsoxl+1e-9).all() and (candidate_cash>=mincash-1e-9).all() and (candidate_beta<=ceiling+1e-9).all()
assert np.max(np.abs(candidate_soxl+p_soxx+candidate_cash-1))<1e-8
feasible=delta>1e-10
x=pd.DataFrame({'trade_date':calendar,'year':calendar.year,'quarter':calendar.quarter,'execute':execute,'binding':binding,'feasible':feasible,
                'delta_beta':delta,'beta_slack':beta_slack,'cash_slack':cash_slack,'soxl_slack':soxl_slack,
                'projected_soxl':p_soxl,'projected_cash':p_cash,'candidate_soxl':candidate_soxl,'candidate_cash':candidate_cash})
def cohort(g):
 e=g[g.execute];f=e[e.feasible]
 return {'sessions':len(g),'execute':len(e),'HOLD':len(g)-len(e),'binding_execute':int(e.binding.sum()),'nonbinding_execute':int((~e.binding).sum()),
         'feasible':len(f),'binding_feasible':int(f.binding.sum()),'nonbinding_feasible':int((~f.binding).sum()),
         'zero_SOXL_headroom_execute':int((e.soxl_slack<=1e-10).sum()),'zero_cash_headroom_execute':int((e.cash_slack<=1e-10).sum()),
         'zero_beta_headroom_execute':int((e.beta_slack<=1e-10).sum()),
         'mean_delta_beta_execute':float(e.delta_beta.mean()) if len(e) else None,'max_delta_beta':float(g.delta_beta.max())}
all_days=cohort(x)
annual=[{'year':int(y),'partial_endpoint':y in (2023,2026),**cohort(g)} for y,g in x.groupby('year',sort=True)]
quarter=[{'year':int(y),'quarter':int(q),**cohort(g)} for (y,q),g in x.groupby(['year','quarter'],sort=True)]
assert len(annual)==4 and len(quarter)==16 and sum(a['sessions'] for a in annual)==944
by_binding={str(v).lower():cohort(g) for v,g in x.groupby('binding')}
windows=[]
for i in range(819):
 g=x.iloc[i:i+126];rw=rolling.iloc[i]
 assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 outcome='weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie'
 w=cohort(g)
 windows.append([str(g.trade_date.iloc[0].date()),outcome,w['feasible'],w['binding_feasible'],w['nonbinding_feasible'],w['zero_SOXL_headroom_execute']])
assert len(windows)==819
w=pd.DataFrame(windows,columns=['start','outcome','feasible','binding_feasible','nonbinding_feasible','zero_SOXL_headroom_execute'])
assert w.outcome.value_counts().to_dict()=={'weak':564,'good':255}
window_summary={name:{'windows':len(g),'mean_feasible':float(g.feasible.mean()),'mean_binding_feasible':float(g.binding_feasible.mean()),
                      'mean_nonbinding_feasible':float(g.nonbinding_feasible.mean()),'zero_nonbinding_feasible_windows':int((g.nonbinding_feasible==0).sum())} for name,g in w.groupby('outcome')}
result={'status':'read_only_exact_f696_cash_to_SOXL_legal_headroom_no_policy','plan_sha256':sha(plan_path),
        'A0_recorder_id':s['A0_recorder_id'],'account_recorder_id':s['account_recorder_id'],
        'all_days':all_days,'annual':annual,'quarter':quarter,'by_binding':by_binding,'rolling126_columns':list(w.columns),'rolling126':windows,'rolling126_summary':window_summary,
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['Legal target headroom is not a filled whole-share trade, fee-net local value or funded own-state account','Binding cap+0.10 is a diagnostic upper envelope, not an approved permanent risk policy','Weak/good original-account windows are hindsight and overlap heavily']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'all_days':all_days,'annual':[(z['year'],z['feasible'],z['binding_feasible'],z['nonbinding_feasible']) for z in annual],
       'windows':window_summary})


{'all_days': {'sessions': 944, 'execute': 797, 'HOLD': 147, 'binding_execute': 261, 'nonbinding_execute': 536, 'feasible': 285, 'binding_feasible': 261, 'nonbinding_feasible': 24, 'zero_SOXL_headroom_execute': 53, 'zero_cash_headroom_execute': 0, 'zero_beta_headroom_execute': 459, 'mean_delta_beta_execute': 0.034489951152054464, 'max_delta_beta': 0.1}, 'annual': [(2023, 52, 52, 0), (2024, 62, 50, 12), (2025, 85, 74, 11), (2026, 86, 85, 1)], 'windows': {'good': {'windows': 255, 'mean_feasible': 34.4078431372549, 'mean_binding_feasible': 28.701960784313727, 'mean_nonbinding_feasible': 5.705882352941177, 'zero_nonbinding_feasible_windows': 124}, 'weak': {'windows': 564, 'mean_feasible': 37.35992907801418, 'mean_binding_feasible': 34.783687943262414, 'mean_nonbinding_feasible': 2.576241134751773, 'zero_nonbinding_feasible_windows': 468}}}


## 中文结论

# 精确f696现金转SOXL：合法开盘头寸空间几乎只在受限层

只读封存精确f696原A_0 0bp Tiered账户的944日、797个开盘EXECUTE/147个HOLD、原提案/合法投影、当前自身beta上限与819个连续126日SOXX相对窗口。逐件验证Qlib Recorder SHA、NAS回读、Qlib日历、原目标资金合计和 `SOXL<=30%/现金>=15%/原beta<=cap`。诊断动作仅在原EXECUTE日把最多0.10 beta从现金转给SOXL；原受限日允许诊断上限cap+0.10，其他日保持原cap；候选再次严格守30% SOXL与15%现金。0新行情、模型、费用标签或资金账户。

目标层面只有 **285/797** 个原执行日存在正的合法SOXL增配容量：原受限 **261/261**，原未受限 **24/536**。未受限536日中 **459日beta剩余容量为0**、**53日SOXL权重余量为0**，现金底线0日阻断；平均可转beta仅 **0.00259**，相比受限日每个可转 **0.10**。2023部分年未受限合法增配 **0日**，2024/2025完整年 **12/11日**，2026部分年 **1日**。全部16季度及819窗逐窗的正容量计数保存在 `result.json`。

原账户未胜SOXX的564个滚动窗中，**468窗**没有任何未受限层SOXL增配日；255个胜窗中为124窗。弱窗平均未受限可增配日仅 **2.58/126**，受限为 **34.78/126**。这与原账户静态现金分解的双层拖累相对照：**这一种固定“原09:46目标后现金转SOXL、原非受限cap不变”的动作，直接覆盖不到大部分非受限现金机会问题**。它不证明一切单层或自身反馈策略必败，也不能将事后弱窗当成交易门槛。

决定 `read_only_exact_f696_SOXL_opening_action_cannot_cover_both_cash_layers_no_policy`。不为这一动作继续拟合预测器或跑完整候选账户；下一机制须在**09:29原始配置意图与09:46自身风险容量**两处同时有明确作用，或提供实质不同的合法动作空间，并先冻结来源、标签及账户规则。目标头寸可行性不是整股成交、费用后局部收益、完整自身状态账户或SOXX净超额；2/5bp和独立前向仍未达成。
